# NMF on ModulAir 00677

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00677
data = pd.read_csv('MOD-00677.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-30T02:01:54Z,576295658,2025-12-29T21:01:54Z,MOD-00677,38.3,1.9,1.406,0.240,0.072,0.025,0.011,...,33.070,30.471,14304.0,14305.0,14306.0,14417.0,14416.0,14420.0,14423.0,5.40
2025-12-30T02:00:54Z,576295660,2025-12-29T21:00:54Z,MOD-00677,37.8,2.0,1.506,0.250,0.084,0.007,0.015,...,33.099,31.187,14304.0,14305.0,14306.0,14417.0,14416.0,14420.0,14423.0,9.71
2025-12-30T01:59:54Z,576295659,2025-12-29T20:59:54Z,MOD-00677,37.3,2.0,1.327,0.223,0.097,0.007,0.015,...,32.646,31.564,14304.0,14305.0,14306.0,14417.0,14416.0,14420.0,14423.0,10.61
2025-12-30T01:58:54Z,576295657,2025-12-29T20:58:54Z,MOD-00677,36.8,2.0,1.364,0.234,0.080,0.010,0.017,...,32.904,31.616,14304.0,14305.0,14306.0,14417.0,14416.0,14420.0,14423.0,10.13
2025-12-30T01:57:54Z,576293842,2025-12-29T20:57:54Z,MOD-00677,36.3,2.0,1.279,0.255,0.103,0.027,0.014,...,31.740,31.343,14304.0,14305.0,14306.0,14417.0,14416.0,14420.0,14423.0,14.81


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-30T02:01:54Z,2025-12-29T21:01:54Z,683.923,1.886,33.070,30.471,1.406,0.240,0.072,0.025,0.011,0.018
2025-12-30T02:00:54Z,2025-12-29T21:00:54Z,686.849,1.822,33.099,31.187,1.506,0.250,0.084,0.007,0.015,0.011
2025-12-30T01:59:54Z,2025-12-29T20:59:54Z,680.225,1.951,32.646,31.564,1.327,0.223,0.097,0.007,0.015,0.018
2025-12-30T01:58:54Z,2025-12-29T20:58:54Z,680.170,1.953,32.904,31.616,1.364,0.234,0.080,0.010,0.017,0.011
2025-12-30T01:57:54Z,2025-12-29T20:57:54Z,677.241,1.823,31.740,31.343,1.279,0.255,0.103,0.027,0.014,0.010


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29T21:01:54Z,683.923,1.886,33.070,30.471,1.406,0.240,0.072,0.025,0.011,0.018
1,2025-12-29T21:00:54Z,686.849,1.822,33.099,31.187,1.506,0.250,0.084,0.007,0.015,0.011
2,2025-12-29T20:59:54Z,680.225,1.951,32.646,31.564,1.327,0.223,0.097,0.007,0.015,0.018
3,2025-12-29T20:58:54Z,680.170,1.953,32.904,31.616,1.364,0.234,0.080,0.010,0.017,0.011
4,2025-12-29T20:57:54Z,677.241,1.823,31.740,31.343,1.279,0.255,0.103,0.027,0.014,0.010


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29 21:01:54,683.923,1.886,33.070,30.471,1.406,0.240,0.072,0.025,0.011,0.018
1,2025-12-29 21:00:54,686.849,1.822,33.099,31.187,1.506,0.250,0.084,0.007,0.015,0.011
2,2025-12-29 20:59:54,680.225,1.951,32.646,31.564,1.327,0.223,0.097,0.007,0.015,0.018
3,2025-12-29 20:58:54,680.170,1.953,32.904,31.616,1.364,0.234,0.080,0.010,0.017,0.011
4,2025-12-29 20:57:54,677.241,1.823,31.740,31.343,1.279,0.255,0.103,0.027,0.014,0.010


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-10-24 10:00:00,740.11,24.23,28.82,2.37,1.90,0.30,0.13,0.03,0.04,0.02
1,2025-10-24 11:00:00,683.13,25.88,30.62,2.59,1.23,0.19,0.08,0.02,0.02,0.02
2,2025-10-24 12:00:00,655.71,27.23,33.30,2.41,0.87,0.13,0.05,0.01,0.01,0.01
3,2025-10-24 13:00:00,658.66,28.75,34.89,2.34,1.52,0.18,0.06,0.01,0.01,0.01
4,2025-10-24 14:00:00,653.92,28.84,36.54,2.25,0.79,0.13,0.05,0.01,0.01,0.01
...,...,...,...,...,...,...,...,...,...,...,...
1581,2025-12-29 17:00:00,665.76,30.38,31.64,2.20,1.37,0.27,0.11,0.02,0.02,0.01
1582,2025-12-29 18:00:00,678.24,31.42,30.51,2.01,1.55,0.29,0.11,0.02,0.02,0.01
1583,2025-12-29 19:00:00,668.57,31.07,30.25,1.93,1.33,0.27,0.10,0.02,0.02,0.01
1584,2025-12-29 20:00:00,676.10,32.08,31.10,1.89,1.24,0.24,0.09,0.02,0.02,0.01


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-10-24 10:00:00,740.11,24.23,28.82,2.37,1.90,0.30,0.13,0.03,0.04,0.02
2025-10-24 11:00:00,683.13,25.88,30.62,2.59,1.23,0.19,0.08,0.02,0.02,0.02
2025-10-24 12:00:00,655.71,27.23,33.30,2.41,0.87,0.13,0.05,0.01,0.01,0.01
2025-10-24 13:00:00,658.66,28.75,34.89,2.34,1.52,0.18,0.06,0.01,0.01,0.01
2025-10-24 14:00:00,653.92,28.84,36.54,2.25,0.79,0.13,0.05,0.01,0.01,0.01


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-10-24 10:00:00,0.518651,0.554716,0.631049,0.041969,0.033310,0.011723,0.013118,0.012,0.020202,0.012270
2025-10-24 11:00:00,0.478721,0.592491,0.670462,0.045865,0.021564,0.007425,0.008073,0.008,0.010101,0.012270
2025-10-24 12:00:00,0.459506,0.623397,0.729144,0.042678,0.015252,0.005080,0.005045,0.004,0.005051,0.006135
2025-10-24 13:00:00,0.461573,0.658196,0.763959,0.041438,0.026648,0.007034,0.006054,0.004,0.005051,0.006135
2025-10-24 14:00:00,0.458251,0.660256,0.800088,0.039844,0.013850,0.005080,0.005045,0.004,0.005051,0.006135


In [11]:
data.to_csv(r'MOD-00677_timeseries_hourly_scaled.csv')

## Implementing NMF

In [12]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [13]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-10-24 10:00:00,0.417716,0.625676,0.626310,0.030890,0.040562,0.015534,0.016095,0.017211,0.028701,0.025522
2025-10-24 11:00:00,0.419568,0.634964,0.667073,0.027876,0.027530,0.006586,0.007808,0.009575,0.021457,0.021185
2025-10-24 12:00:00,0.424723,0.648779,0.726745,0.024308,0.022179,0.001245,0.001725,0.003162,0.014944,0.017398
2025-10-24 13:00:00,0.441039,0.673732,0.762133,0.024074,0.031138,0.003557,0.002047,0.002238,0.013180,0.016371
2025-10-24 14:00:00,0.439275,0.674457,0.798416,0.022570,0.021095,0.000179,0.000248,0.001439,0.013969,0.017478
...,...,...,...,...,...,...,...,...,...,...
2025-12-29 17:00:00,0.461878,0.698979,0.692134,0.031099,0.029805,0.002945,0.003490,0.005168,0.016481,0.017972
2025-12-29 18:00:00,0.476254,0.718287,0.667965,0.033857,0.033035,0.002759,0.003082,0.004748,0.015417,0.016933
2025-12-29 19:00:00,0.470167,0.709606,0.662404,0.033347,0.030376,0.001805,0.002501,0.004401,0.015205,0.016780


In [14]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.092482,0.005167,0.020557,0.113775
1,0.099357,0.001254,0.011542,0.109858
2,0.108520,0.000000,0.002897,0.103145
3,0.113252,0.002741,0.000003,0.105092
4,0.119222,0.000000,0.000416,0.098103
...,...,...,...,...
1508,0.103238,0.000562,0.005158,0.130472
1509,0.099600,0.000706,0.004291,0.143400
1510,0.098912,0.000000,0.004201,0.141266
1511,0.101758,0.000000,0.002698,0.144552


In [15]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [16]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,2.244579,3.600040,6.696884,0.000000,0.000000,0.000000,0.000000,0.001763,0.109845,0.144539
Feature 2,1.113436,1.218844,1.348946,0.000000,3.115707,1.297165,0.745971,0.373284,0.066797,0.000000
Feature 3,0.263319,0.100147,0.000000,0.234837,0.000000,0.429629,0.595449,0.682124,0.856044,0.591259
Feature 4,1.748788,2.499516,0.000000,0.229071,0.215030,0.000000,0.000000,0.009638,0.005269,0.000000


In [17]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-10-24 10:00:00,0.092482,0.005167,0.020557,0.113775
2025-10-24 11:00:00,0.099357,0.001254,0.011542,0.109858
2025-10-24 12:00:00,0.108520,0.000000,0.002897,0.103145
2025-10-24 13:00:00,0.113252,0.002741,0.000003,0.105092
2025-10-24 14:00:00,0.119222,0.000000,0.000416,0.098103
...,...,...,...,...
2025-12-29 17:00:00,0.103238,0.000562,0.005158,0.130472
2025-12-29 18:00:00,0.099600,0.000706,0.004291,0.143400
2025-12-29 19:00:00,0.098912,0.000000,0.004201,0.141266


In [18]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,2.244579,3.600040,6.696884,0.000000,0.000000,0.000000,0.000000,0.001763,0.109845,0.144539
Factor 2,1.113436,1.218844,1.348946,0.000000,3.115707,1.297165,0.745971,0.373284,0.066797,0.000000
Factor 3,0.263319,0.100147,0.000000,0.234837,0.000000,0.429629,0.595449,0.682124,0.856044,0.591259
Factor 4,1.748788,2.499516,0.000000,0.229071,0.215030,0.000000,0.000000,0.009638,0.005269,0.000000


In [19]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.374219,0.061898,0.015055,0.545955,0.002873
no,0.000000,0.000000,0.164536,0.876345,-0.040882
no2,0.413373,0.046666,0.003944,0.537427,-0.001409
o3,0.936999,0.062933,0.000000,0.000000,0.000068
bin0,0.000000,0.722990,0.000000,0.280211,-0.003201
bin1,0.000000,0.813580,0.277140,0.000000,-0.090720
bin2,0.000000,0.573020,0.470428,0.000000,-0.043448
bin3,0.004681,0.330445,0.621046,0.047912,-0.004083
bin4,0.251713,0.051039,0.672730,0.022610,0.001907
bin5,0.432234,0.000000,0.606360,0.000000,-0.038595


In [20]:
results.to_csv(r'4_factor_results.csv')
comp.to_csv(r'4_factor_comp.csv')
res.to_csv(r'4_factor_resid.csv')